# 1. Swing Unit Map
หน่วยที่ margin ชนะ/แพ้น้อย = battleground ที่ต้องทุ่มเทหาเสียง

In [4]:
import sys
!{sys.executable} -m pip install folium -q

In [ ]:
import pandas as pd
import folium
from pathlib import Path

CLEAN = Path('../cleaned/')
results = pd.read_csv(CLEAN / 'master_results_cleaned.csv')
coords  = pd.read_csv(CLEAN / 'coords_cleaned.csv')

JOIN = ['district', 'sub-district', 'unit_number']
r = results[(results['unit_number'] != -1) & (results['type'] == 'เขต')]

# คะแนนรายหน่วยรายผู้สมัคร
unit_score = r.groupby(JOIN + ['name'])['score'].sum().reset_index()

# อันดับ 1 และ 2 ในแต่ละหน่วย
ranked = (
    unit_score.sort_values(JOIN + ['score'], ascending=[True,True,True,False])
    .groupby(JOIN)
)
top1 = ranked.nth(0).rename(columns={'name':'winner','score':'score1'})
top2 = ranked.nth(1).rename(columns={'name':'runner_up','score':'score2'})

swing = top1.merge(top2[JOIN + ['runner_up','score2']], on=JOIN)
swing['margin'] = swing['score1'] - swing['score2']
swing['margin_pct'] = (swing['margin'] / swing['score1'] * 100).round(1)

# merge พิกัด
df = swing.merge(coords, on=JOIN, how='inner')

# สี: margin น้อย = แดงเข้ม, มาก = เขียว
import numpy as np
def margin_color(pct):
    if pct < 10:   return '#e74c3c'   # battleground
    elif pct < 25: return '#f39c12'   # contested
    elif pct < 50: return '#f1c40f'   # leaning
    else:          return '#2ecc71'   # safe

df['color'] = df['margin_pct'].apply(margin_color)

center = [df['latitude'].mean(), df['longitude'].mean()]
m = folium.Map(location=center, zoom_start=9, tiles='CartoDB positron')

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color=row['color'], fill=True, fill_color=row['color'], fill_opacity=0.8,
        popup=folium.Popup(
            f"<b>{row['district']} ต.{row['sub-district']} หน่วย {int(row['unit_number'])}</b><br>"
            f"ผู้ชนะ: {row['winner']} ({int(row['score1']):,})<br>"
            f"รองชนะ: {row['runner_up']} ({int(row['score2']):,})<br>"
            f"Margin: {int(row['margin']):,} ({row['margin_pct']}%)",
            max_width=280
        ),
        tooltip=f"margin {row['margin_pct']}% — {row['winner']}"
    ).add_to(m)

legend_html = """
<div style="position:fixed;bottom:40px;left:40px;z-index:1000;background:white;
     padding:12px 16px;border-radius:8px;box-shadow:2px 2px 8px rgba(0,0,0,0.3);
     font-family:sans-serif;font-size:13px;">
  <b>Swing Zone</b><br><br>
  <span style="background:#e74c3c;padding:2px 10px;border-radius:3px;">&nbsp;</span> &lt;10% — Battleground<br>
  <span style="background:#f39c12;padding:2px 10px;border-radius:3px;">&nbsp;</span> 10–25% — Contested<br>
  <span style="background:#f1c40f;padding:2px 10px;border-radius:3px;">&nbsp;</span> 25–50% — Leaning<br>
  <span style="background:#2ecc71;padding:2px 10px;border-radius:3px;">&nbsp;</span> &gt;50% — Safe
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))

print(f'Battleground units (<10% margin): {(df["margin_pct"]<10).sum()}')
print(f'Contested units (10-25%): {((df["margin_pct"]>=10)&(df["margin_pct"]<25)).sum()}')
print(f'Safe units (>50%): {(df["margin_pct"]>50).sum()}')
m


Battleground units (<10% margin): 49
Contested units (10-25%): 42
Safe units (>50%): 118


: 